# Qwen resume smoke: visual review

Run this notebook from the repository's `notebooks/` directory on CARC after the smoke job finishes, or copy the smoke `merged`, `records`, and `images` directories into the same repository layout on another machine. It verifies the merged smoke contract and displays every clean page beside its saved adversarial PNG and an amplified absolute-difference image.

Production acceptance requires: 10 complete records, no failures, unchanged dimensions, pixel distance within epsilon, exact official-forward preprocessing, reasonable perceptual metrics, and no visible corruption or page mismatch.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'configs').is_dir():
    raise RuntimeError('Open this notebook from the poisoned-paperwork repository or notebooks directory')
sys.path.insert(0, str(ROOT / 'src'))

from data import load_processed_rows, row_images

RUN_ID = 'qwen_resume_smoke_shail_v2'
RUN = ROOT / 'outputs' / 'qwen_attack' / RUN_ID
RECORD_DIR = RUN / 'shards' / 'worker_00' / 'records'
SUMMARY_PATH = RUN / 'merged' / 'summary.json'

record_paths = sorted(RECORD_DIR.glob('*.json'))
if not record_paths:
    raise FileNotFoundError(f'No smoke records found under {RECORD_DIR}')
records = [json.loads(path.read_text(encoding='utf-8')) for path in record_paths]
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
rows = load_processed_rows(ROOT, 'validation', {r['record_id'] for r in records})
print(f'Loaded {len(records)} records from {RUN_ID}')
summary

In [ ]:
metrics = pd.DataFrame([{
    'record_id': r['record_id'],
    'pages': len(r['images']),
    'clean': r['clean_target'],
    'target': r['attack_target'],
    'prediction': r['prediction_normalized'],
    'success': bool(r['saved_image_success']),
    'steps': r['completed_steps'],
    'linf_255': r['linf_255'],
    'lpips': r['lpips'],
    'ssim': r['ssim'],
    'peak_gpu_gib': r['peak_gpu_memory_mb'] / 1024,
    'resumed': bool(r['checkpoint_resumed']),
    'official_forward_max_error': r['processor_max_abs_difference'],
    'surrogate_mean_error': r['processor_surrogate_mean_abs_difference'],
} for r in records]).sort_values('record_id')

display(metrics.style.format({
    'linf_255': '{:.6f}', 'lpips': '{:.5f}', 'ssim': '{:.5f}',
    'peak_gpu_gib': '{:.2f}', 'official_forward_max_error': '{:.2e}',
    'surrogate_mean_error': '{:.6f}',
}))

## Visual comparison

The third panel multiplies the absolute pixel difference by 16 so that small perturbations become visible. It is diagnostic only; it is not the attacked image shown to the model.

In [ ]:
visual_checks = []
for record in sorted(records, key=lambda r: r['record_id']):
    clean_pages = [image.convert('RGB') for image in row_images(rows[record['record_id']])]
    attacked_pages = []
    for image_info in record['images']:
        recorded_path = Path(image_info['path'])
        path = recorded_path
        if not path.is_file():
            path = RUN / 'shards' / 'worker_00' / 'images' / recorded_path.parent.name / recorded_path.name
        if not path.is_file():
            raise FileNotFoundError(f'Neither recorded nor portable result path exists: {recorded_path} / {path}')
        with Image.open(path) as source:
            attacked_pages.append(source.convert('RGB').copy())
    if len(clean_pages) != len(attacked_pages):
        raise AssertionError(f"Page-count mismatch for {record['record_id']}")

    figure, axes = plt.subplots(len(clean_pages), 3, figsize=(15, 6 * len(clean_pages)), squeeze=False)
    for page_index, (clean, attacked) in enumerate(zip(clean_pages, attacked_pages)):
        if clean.size != attacked.size:
            raise AssertionError(f"Dimension mismatch for {record['record_id']} page {page_index}")
        clean_array = np.asarray(clean, dtype=np.int16)
        attacked_array = np.asarray(attacked, dtype=np.int16)
        difference = np.abs(attacked_array - clean_array)
        actual_linf = int(difference.max())
        visual_checks.append({
            'record_id': record['record_id'], 'page': page_index,
            'width': clean.width, 'height': clean.height,
            'actual_linf_255': actual_linf,
        })
        axes[page_index, 0].imshow(clean)
        axes[page_index, 0].set_title(f'Clean page {page_index}')
        axes[page_index, 1].imshow(attacked)
        axes[page_index, 1].set_title(
            f"Attacked: ε={record['epsilon_255']}, L∞={actual_linf}, "
            f"SSIM={record['images'][page_index]['ssim']:.5f}"
        )
        axes[page_index, 2].imshow(np.clip(difference * 16, 0, 255).astype(np.uint8))
        axes[page_index, 2].set_title('Absolute difference ×16')
        for axis in axes[page_index]:
            axis.axis('off')
    figure.suptitle(
        f"{record['record_id']} | clean={record['clean_target']} | "
        f"target={record['attack_target']} | output={record['prediction_normalized']} | "
        f"success={record['saved_image_success']}", fontsize=13
    )
    plt.tight_layout()
    plt.show()

In [ ]:
checks = pd.DataFrame(visual_checks)
assert summary['records'] == 10
assert summary['unique_documents'] == 10
assert summary['failures_logged'] == 0
assert len(records) == 10
assert all(r['status'] == 'complete' for r in records)
assert all(np.isfinite(r['lpips']) and np.isfinite(r['ssim']) for r in records)
assert all(r['linf_255'] <= r['epsilon_255'] + 1e-4 for r in records)
assert all(r['processor_max_abs_difference'] <= 1e-6 for r in records)
assert (checks['actual_linf_255'] <= 8).all()
print('AUTOMATED SMOKE REVIEW PASSED')
print(f"Maximum GPU memory: {metrics['peak_gpu_gib'].max():.2f} GiB")
print(f"Saved-image targeted successes: {int(metrics['success'].sum())}/{len(metrics)}")
print(f"Checkpoint-resumed records: {int(metrics['resumed'].sum())}/{len(metrics)}")
display(checks)

## Manual acceptance

After running all cells, confirm manually that:

- every attacked page corresponds to the clean page beside it;
- no page is cropped, rotated, blank, color-shifted, or visibly corrupted;
- text remains readable and the perturbation is not obvious at normal zoom;
- the amplified difference resembles distributed low-amplitude noise rather than a loading or alignment error.

Record any rejected `record_id` before enabling production. The successful v2 smoke is immutable; do not edit its images or JSON records.